In [8]:
import collections, numpy as np, pandas as pd, matplotlib, matplotlib.pyplot as plt, seaborn as sns
import floweaver, ipysankeywidget
%load_ext autoreload
%autoreload 2
from af2genomics import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
df_ = pd.read_csv(workpath('24.02.06_Sankey/clinvar_subset.tsv'), delim_whitespace=True)
printlen(df_, 'ClinVar variants')

<module>: 351,349 ClinVar variants


In [10]:
def clinvar_label_(clintype):
    if clintype == 0:
        return 'Benign'
    elif clintype == 1:
        return 'Pathogenic'
    elif clintype == 2:
        return 'VUS'

df_['clinvar_label'] = df_['clintype'].map(clinvar_label_)
df_['clinvar_label'].value_counts()

VUS           245153
Benign         61119
Pathogenic     45077
Name: clinvar_label, dtype: int64

In [11]:
def pathogenicity_label_(eve):
    if eve < .4:
        return 'Pred. benign'
    elif eve >= .7:
        return 'Pred. pathogenic'
    else:
        return 'Pred. ambigous'

df_['pathogenicity_label'] = df_['eve'].map(pathogenicity_label_)
df_['pathogenicity_label'].value_counts()

Pred. benign        192007
Pred. pathogenic    117379
Pred. ambigous       41963
Name: pathogenicity_label, dtype: int64

In [12]:
def mechanistic_label_(r):
    if r.pred_ddg > 2:
        return 'Stability'
    elif r.interface >= .4:
        return 'Interface'
    elif r.pocketscore > 25 & r.pocketrank<=3:
        return 'Pockets'
    else:
        return 'Unassigned'

df_['mechanistic_label'] = df_.apply(mechanistic_label_, axis=1)
df_['mechanistic_label'].value_counts()

Unassigned    206220
Stability      66511
Pockets        42924
Interface      35694
Name: mechanistic_label, dtype: int64

In [13]:
cols_ = ['clinvar_label', 'pathogenicity_label', 'mechanistic_label']
flows_ = df_[cols_].groupby(cols_).size().reset_index().rename({0: 'value',
    'clinvar_label': 'source',
    'pathogenicity_label': 'type',
    'mechanistic_label': 'target',
}, axis=1)
flows_
#ipysankeywidget.SankeyWidget(links=flows_.to_dict('records'))

,source,type,target,value
0,Benign,Pred. ambigous,Interface,773
1,Benign,Pred. ambigous,Pockets,909
2,Benign,Pred. ambigous,Stability,798
3,Benign,Pred. ambigous,Unassigned,3999
4,Benign,Pred. benign,Interface,3575
5,Benign,Pred. benign,Pockets,4716
6,Benign,Pred. benign,Stability,2288
7,Benign,Pred. benign,Unassigned,30209
8,Benign,Pred. pathogenic,Interface,1345
9,Benign,Pred. pathogenic,Pockets,1802


In [14]:
clinvar_part_ = floweaver.Partition.Simple('source', ['Benign', 'VUS', 'Pathogenic'])
pathogenicity_part_ = floweaver.Partition.Simple('type', ['Pred. benign', 'Pred. ambigous', 'Pred. pathogenic'])
mechanistic_part_ = floweaver.Partition.Simple('target', ['Unassigned', 'Stability', 'Interface', 'Pockets'])

nodes_ = {
    'clinvar_col': floweaver.ProcessGroup(selection=flows_['source'].unique().tolist(), title='Known annotations', partition=clinvar_part_),
    'pathogenicity_col': floweaver.Waypoint(pathogenicity_part_, title='EVE'),
    'mechanistic_col': floweaver.ProcessGroup(selection=flows_['target'].unique().tolist(), title='Structural mechanisms', partition=mechanistic_part_),
}

#nodes_['mechanistic_col'].partition = floweaver.Partition.Simple('process', flows_['target'].unique().tolist())
#nodes_['mechanistic_col'].partition = floweaver.Partition.Simple('process', ['Unassigned', 'Stability', 'Interface', 'Pockets'])

ordering_ = [['clinvar_col'], ['pathogenicity_col'], ['mechanistic_col'],]

bundles_ = [floweaver.Bundle('clinvar_col', 'mechanistic_col', waypoints=['pathogenicity_col']),]

palette_ = {
    'Pred. benign': matplotlib.colors.TABLEAU_COLORS['tab:blue'],
    'Pred. ambigous': matplotlib.colors.TABLEAU_COLORS['tab:gray'],
    'Pred. pathogenic': matplotlib.colors.TABLEAU_COLORS['tab:red'],
    #'Unassigned':  matplotlib.colors.TABLEAU_COLORS['tab:purple'],
}

sdd_ = floweaver.SankeyDefinition(nodes_, bundles_, ordering_, flow_partition=pathogenicity_part_)
floweaver.weave(sankey_definition=sdd_, dataset=flows_, palette=palette_).to_widget(width=600, height=400).auto_save_svg('clinvar_mechanisms.svg')

SankeyWidget(groups=[{'id': 'clinvar_col', 'type': 'process', 'title': 'Known annotations', 'nodes': ['clinvar…